# BirdCLEF 2026 — Manual Baseline

Hand-crafted baseline for the project report, used to compare against the autonomous agent's best model.

**Model:** `cnn_small_v1` (3 Conv2D blocks + global pooling + linear head)

**Training:** 3 epochs, Adam optimizer, lr=1e-3, batch_size=32, no augmentation.

The notebook uses the same fixed audio pipeline and data loader as the agent, so the comparison with agent-produced experiments is fair.

## Setup

Run `bash scripts/preprocess.sh` (or `python scripts/build_profile.py --sample 100`) once before executing this notebook, so that `data/processed/dataset_profile.json` and the spectrograms exist.

In [ ]:
import sys, os
# Make sure the repo root is importable when the notebook is launched from elsewhere
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Disable CUDA to match the Kaggle submission environment
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')

import json
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np

In [ ]:
from pipelines.data_loader import load_precomputed_dataset
from agent.models import DatasetProfile

PROFILE_PATH = Path('data/processed/dataset_profile.json')
SPEC_DIR = Path('data/processed/spectrograms')
LABELS_CSV = Path('data/processed/labels.csv')

profile = DatasetProfile.from_json_file(PROFILE_PATH)
print('classes:', profile.num_classes)
print('samples:', profile.num_samples)
print('shape:  ', profile.spectrogram_shape)

train_loader, val_loader, num_classes = load_precomputed_dataset(
    profile_path=PROFILE_PATH,
    spectrograms_dir=SPEC_DIR,
    labels_csv=LABELS_CSV,
    batch_size=32,
    num_workers=0,
)

In [ ]:
class CnnSmallV1(nn.Module):
    """The hand-rolled baseline used in the registry."""
    def __init__(self, num_classes: int, in_channels: int = 1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(64, num_classes)

    def forward(self, x):
        f = self.features(x)
        f = f.view(f.size(0), -1)
        return self.head(f)

model = CnnSmallV1(num_classes=num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()
print(sum(p.numel() for p in model.parameters()), 'parameters')

In [ ]:
from sklearn.metrics import roc_auc_score

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total, losses = 0, 0.0
    for x, y in loader:
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        losses += loss.item() * x.size(0)
        total += x.size(0)
    return losses / max(1, total)

def eval_model(model, loader):
    model.eval()
    all_y, all_p = [], []
    with torch.no_grad():
        for x, y in loader:
            p = torch.sigmoid(model(x))
            all_y.append(y.numpy())
            all_p.append(p.numpy())
    all_y = np.concatenate(all_y)
    all_p = np.concatenate(all_p)
    # Macro ROC-AUC over classes that have at least one positive
    scores = []
    for i in range(all_y.shape[1]):
        if all_y[:, i].sum() > 0:
            scores.append(roc_auc_score(all_y[:, i], all_p[:, i]))
    return float(np.mean(scores)) if scores else 0.0

for epoch in range(3):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_auc = eval_model(model, val_loader)
    print(f'epoch {epoch + 1}: train_loss={train_loss:.4f}  val_roc_auc_macro={val_auc:.4f}')

## Save results for the report

The final `val_roc_auc_macro` from this notebook goes into the report's "manual baseline vs. agent" comparison table.